# TeamError404 | GSM8K batch pilot

Select a GPU in Runtime → Change runtime type. Run the cells in order. Upload the software ZIP from the commit you want to test. The notebook uses the runner and question IDs from that ZIP. Use only one tab for this notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

archives = files.upload()
assert len(archives) == 1, "Upload one software ZIP."
archive = Path(next(iter(archives)))
with zipfile.ZipFile(archive) as bundle:
    names = set(bundle.namelist())
    required = ("runner.py", "question_ids.json", "requirements.txt", "test_runner.py")
    roots = [Path(name).parent for name in sorted(names)
             if name == "runner.py" or name.endswith("/thomas_cannon/runner.py")]
    roots = [root for root in roots if all((root / name).as_posix() in names for name in required)]
    if len(roots) != 1:
        raise ValueError("Upload Cannon's software ZIP or the TeamError404 GitHub ZIP.")
    for name in required:
        Path(name).write_bytes(bundle.read((roots[0] / name).as_posix()))
%pip install -q -r requirements.txt


In [ ]:
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime."
print(torch.cuda.get_device_name(0))
!python -m unittest -v


In [ ]:
output = "/content/drive/MyDrive/TeamError404/pilot-shared-v4"
!python -u runner.py --limit 25 --question-batch-size 4 --output "{output}"
